
# 01 — Fireline QA/QC × NIROPS Joinability Audit

This notebook performs the **first real data audit** for the dynamic wildfire fireline-effectiveness project.

It is intentionally designed to run **before** HRRR, LANDFIRE, terrain extraction, feature engineering, or ML.

## What this notebook does

1. Uses one project-root path as input.
2. Finds the downloaded Fireline QA/QC and NIROPS folders.
3. Inspects ZIP contents **before extraction**.
4. Reports archive sizes and likely geospatial files.
5. Extracts only the core archives needed for the joinability study.
6. Automatically locates candidate shapefiles / GeoPackages / File Geodatabases.
7. Reads the Fireline QA/QC and NIROPS datasets with GeoPandas / Pyogrio.
8. Prints schemas, CRS, geometry types, date coverage, field names, and missingness.
9. Detects and normalizes `IRWINID`.
10. Computes **exact incident overlap by IRWINID**.
11. Audits Fireline engagement outcomes:
   - Held
   - Burned Over
   - Not Engaged
12. Computes line length by engagement class.
13. Counts NIROPS observations per matched incident.
14. Summarizes temporal gaps between NIROPS observations.
15. Builds conservative fallback matching candidates using incident name/year only when exact IRWIN matching is unavailable.
16. Writes reproducible CSVs and a Markdown report to:

```text
<PROJECT_ROOT>/outputs/audit/
```

## Important methodology

- Raw downloads are **never modified**.
- Exact normalized `IRWINID` is the primary join.
- Fuzzy/fallback matches are **not automatically accepted**.
- The notebook distinguishes incident counts from individual line/perimeter records.
- No ML is trained here.


In [ ]:
import os  # Configurable project root for the public release.

from pathlib import Path

# ============================================================
# USER INPUT — CHANGE ONLY THIS PATH IF NEEDED
# ============================================================

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))

# ============================================================
# EXPECTED DOWNLOADED DATA FOLDERS
# ============================================================

FIRELINE_ROOT = PROJECT_ROOT / "Fireline_QAQC_2017_2024"
NIROPS_ROOT = PROJECT_ROOT / "NIROPS_2020_2024"

# Working/extracted data created by THIS notebook.
WORK_ROOT = PROJECT_ROOT / "data" / "interim" / "joinability_audit"
FIRELINE_EXTRACT = WORK_ROOT / "fireline_extracted"
NIROPS_EXTRACT = WORK_ROOT / "nirops_extracted"

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "audit"

for p in [WORK_ROOT, FIRELINE_EXTRACT, NIROPS_EXTRACT, OUTPUT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT    :", PROJECT_ROOT)
print("FIRELINE_ROOT   :", FIRELINE_ROOT)
print("NIROPS_ROOT     :", NIROPS_ROOT)
print("WORK_ROOT       :", WORK_ROOT)
print("OUTPUT_ROOT     :", OUTPUT_ROOT)

assert FIRELINE_ROOT.exists(), f"Missing folder: {FIRELINE_ROOT}"
assert NIROPS_ROOT.exists(), f"Missing folder: {NIROPS_ROOT}"



## Install / import dependencies

The audit uses:

- `pandas`
- `geopandas`
- `shapely`
- `pyogrio`
- `pyproj`
- `rapidfuzz`
- `tqdm`

`pyogrio` is useful for inspecting File Geodatabases and listing layers.


In [ ]:

import sys
import subprocess
import importlib.util

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "geopandas": "geopandas",
    "shapely": "shapely",
    "pyogrio": "pyogrio",
    "pyproj": "pyproj",
    "rapidfuzz": "rapidfuzz",
    "tqdm": "tqdm",
}

for module_name, package_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing {package_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

import os
import re
import json
import math
import shutil
import zipfile
import hashlib
import warnings
from collections import Counter, defaultdict
from datetime import datetime, timezone
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
from shapely.geometry import LineString, MultiLineString
from rapidfuzz import fuzz, process
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)

print("Dependencies ready.")
print("Python:", sys.executable)
print("GeoPandas:", gpd.__version__)
print("Pyogrio:", pyogrio.__version__)



# 1. Discover downloaded archives

The notebook first inventories the downloaded folders and identifies ZIP files without extracting them.


In [ ]:

def human_bytes(n):
    n = float(n)
    units = ["B", "KB", "MB", "GB", "TB"]
    for unit in units:
        if n < 1024 or unit == units[-1]:
            return f"{n:.2f} {unit}"
        n /= 1024

def folder_size(path):
    path = Path(path)
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())

def inventory_folder(root):
    root = Path(root)
    rows = []
    for p in sorted(root.rglob("*")):
        if p.is_file():
            rows.append({
                "relative_path": str(p.relative_to(root)),
                "suffix": p.suffix.lower(),
                "size_bytes": p.stat().st_size,
                "size": human_bytes(p.stat().st_size),
            })
    return pd.DataFrame(rows)

fireline_inventory = inventory_folder(FIRELINE_ROOT)
nirops_inventory = inventory_folder(NIROPS_ROOT)

print("Fireline folder size:", human_bytes(folder_size(FIRELINE_ROOT)))
display(fireline_inventory)

print("\nNIROPS folder size:", human_bytes(folder_size(NIROPS_ROOT)))
display(nirops_inventory)



# 2. Inspect ZIP contents before extraction

This stage reads archive directories only. It does not extract the files.

For each ZIP, the notebook reports:

- file count
- compressed size
- uncompressed size
- compression ratio
- likely geospatial components
- nested ZIP files


In [ ]:

GEO_SUFFIXES = {
    ".shp", ".shx", ".dbf", ".prj", ".cpg",
    ".gpkg", ".geojson", ".json",
    ".tif", ".tiff", ".csv", ".parquet"
}

def zip_inventory(zip_path):
    zip_path = Path(zip_path)
    rows = []
    with zipfile.ZipFile(zip_path, "r") as zf:
        for info in zf.infolist():
            if info.is_dir():
                continue
            suffix = Path(info.filename).suffix.lower()
            rows.append({
                "archive": zip_path.name,
                "path": info.filename,
                "suffix": suffix,
                "compressed_bytes": info.compress_size,
                "uncompressed_bytes": info.file_size,
                "compressed": human_bytes(info.compress_size),
                "uncompressed": human_bytes(info.file_size),
                "likely_geospatial": suffix in GEO_SUFFIXES or ".gdb/" in info.filename.lower(),
                "nested_zip": suffix == ".zip",
            })
    return pd.DataFrame(rows)

def summarize_zip(zip_path):
    df = zip_inventory(zip_path)
    compressed = df["compressed_bytes"].sum()
    uncompressed = df["uncompressed_bytes"].sum()
    ratio = (uncompressed / compressed) if compressed else np.nan
    print("=" * 90)
    print(zip_path)
    print(f"Archive on disk       : {human_bytes(Path(zip_path).stat().st_size)}")
    print(f"Contained file count  : {len(df):,}")
    print(f"Compressed payload    : {human_bytes(compressed)}")
    print(f"Uncompressed payload  : {human_bytes(uncompressed)}")
    print(f"Expansion ratio       : {ratio:.2f}x" if np.isfinite(ratio) else "Expansion ratio: n/a")
    print(f"Likely geospatial     : {int(df['likely_geospatial'].sum()):,}")
    print(f"Nested ZIPs           : {int(df['nested_zip'].sum()):,}")
    return df

fireline_zips = sorted(FIRELINE_ROOT.rglob("*.zip"))
nirops_zips = sorted(NIROPS_ROOT.rglob("*.zip"))

print("FIRELINE ZIP FILES")
for p in fireline_zips:
    print(" -", p, human_bytes(p.stat().st_size))

print("\nNIROPS ZIP FILES")
for p in nirops_zips:
    print(" -", p, human_bytes(p.stat().st_size))


In [ ]:

fireline_zip_tables = {}
for z in fireline_zips:
    try:
        df = summarize_zip(z)
        fireline_zip_tables[str(z)] = df
        display(df[df["likely_geospatial"] | df["nested_zip"]].head(200))
    except Exception as exc:
        print("Could not inspect:", z, exc)

nirops_zip_tables = {}
for z in nirops_zips:
    try:
        df = summarize_zip(z)
        nirops_zip_tables[str(z)] = df
        display(df[df["likely_geospatial"] | df["nested_zip"]].head(200))
    except Exception as exc:
        print("Could not inspect:", z, exc)



# 3. Select the core archives

Expected files based on the downloaded release:

### Fireline
```text
RDS-2025-0011.zip
```

### NIROPS
```text
NIROPS_2020_2024_R1_R6.zip
```

The code searches by filename rather than assuming an exact nested folder path.


In [ ]:

def choose_named_zip(root, preferred_names):
    root = Path(root)
    all_zips = list(root.rglob("*.zip"))
    lower_map = {p.name.lower(): p for p in all_zips}

    for name in preferred_names:
        if name.lower() in lower_map:
            return lower_map[name.lower()]

    # Partial match fallback.
    for p in all_zips:
        name_low = p.name.lower()
        for wanted in preferred_names:
            stem = Path(wanted).stem.lower()
            if stem in name_low:
                return p

    return None

FIRELINE_CORE_ZIP = choose_named_zip(
    FIRELINE_ROOT,
    ["RDS-2025-0011.zip"]
)

NIROPS_CORE_ZIP = choose_named_zip(
    NIROPS_ROOT,
    ["NIROPS_2020_2024_R1_R6.zip"]
)

print("Fireline core ZIP:", FIRELINE_CORE_ZIP)
print("NIROPS core ZIP :", NIROPS_CORE_ZIP)

assert FIRELINE_CORE_ZIP is not None, "Could not locate RDS-2025-0011.zip"
assert NIROPS_CORE_ZIP is not None, "Could not locate NIROPS_2020_2024_R1_R6.zip"



# 4. Extract the core datasets

Extraction occurs into:

```text
data/interim/joinability_audit/
```

The original ZIP files remain untouched.

The extraction function is resumable: if a completion marker exists, it will not extract again.


In [ ]:

def extract_zip_once(zip_path, destination):
    zip_path = Path(zip_path)
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    marker = destination / ".extraction_complete.json"

    if marker.exists():
        print(f"SKIP: already extracted -> {destination}")
        return destination

    print(f"Extracting:\n  {zip_path}\n-> {destination}")
    print(f"Archive size: {human_bytes(zip_path.stat().st_size)}")

    with zipfile.ZipFile(zip_path, "r") as zf:
        members = zf.infolist()
        total_uncompressed = sum(i.file_size for i in members if not i.is_dir())
        print("Expected uncompressed size:", human_bytes(total_uncompressed))

        for info in tqdm(members, desc=zip_path.name, unit="file"):
            zf.extract(info, destination)

    marker.write_text(json.dumps({
        "source_zip": str(zip_path),
        "extracted_at_utc": datetime.now(timezone.utc).isoformat(),
        "archive_size_bytes": zip_path.stat().st_size,
        "output_size_bytes": folder_size(destination),
    }, indent=2), encoding="utf-8")

    print("Extracted size:", human_bytes(folder_size(destination)))
    return destination

extract_zip_once(FIRELINE_CORE_ZIP, FIRELINE_EXTRACT)
extract_zip_once(NIROPS_CORE_ZIP, NIROPS_EXTRACT)



# 5. Discover spatial datasets after extraction

The code looks for:

- Shapefiles (`.shp`)
- GeoPackages (`.gpkg`)
- GeoJSON
- File Geodatabases (`.gdb` directories)

File Geodatabase layers are listed using `pyogrio.list_layers()`.


In [ ]:

def discover_spatial_sources(root):
    root = Path(root)
    sources = []

    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in {".shp", ".gpkg", ".geojson"}:
            sources.append({
                "kind": p.suffix.lower().lstrip("."),
                "path": p,
                "layer": None,
                "size_bytes": p.stat().st_size,
            })

        elif p.is_dir() and p.suffix.lower() == ".gdb":
            try:
                layers = pyogrio.list_layers(p)
                for layer_name, geom_type in layers:
                    sources.append({
                        "kind": "gdb",
                        "path": p,
                        "layer": layer_name,
                        "geometry_type_hint": geom_type,
                        "size_bytes": folder_size(p),
                    })
            except Exception as exc:
                print(f"WARNING: could not list GDB layers in {p}: {exc}")

    return pd.DataFrame(sources)

fireline_sources = discover_spatial_sources(FIRELINE_EXTRACT)
nirops_sources = discover_spatial_sources(NIROPS_EXTRACT)

print("FIRELINE SPATIAL SOURCES")
display(fireline_sources)

print("\nNIROPS SPATIAL SOURCES")
display(nirops_sources)



# 6. Inspect candidate schemas without loading everything

We inspect field names and a small sample first. This avoids unnecessarily loading a very large layer before identifying the correct source.


In [ ]:

def read_source(path, layer=None, max_features=None, columns=None):
    kwargs = {}
    if layer is not None:
        kwargs["layer"] = layer
    if max_features is not None:
        kwargs["max_features"] = max_features
    if columns is not None:
        kwargs["columns"] = columns

    return gpd.read_file(path, engine="pyogrio", **kwargs)

def inspect_source_row(row, sample_n=5):
    path = row["path"]
    layer = row.get("layer")
    print("=" * 90)
    print("PATH :", path)
    print("LAYER:", layer)
    try:
        gdf = read_source(path, layer=layer, max_features=sample_n)
        print("CRS           :", gdf.crs)
        print("Geometry      :", list(gdf.geom_type.unique()) if len(gdf) else [])
        print("Columns       :", list(gdf.columns))
        print("Sample rows   :", len(gdf))
        display(gdf.head(sample_n).drop(columns="geometry", errors="ignore"))
        return gdf
    except Exception as exc:
        print("ERROR:", exc)
        return None

print("---- FIRELINE CANDIDATES ----")
for _, row in fireline_sources.iterrows():
    inspect_source_row(row)

print("\n---- NIROPS CANDIDATES ----")
for _, row in nirops_sources.iterrows():
    inspect_source_row(row)



# 7. Automatically select likely Fireline and NIROPS layers

Selection is based on field-name evidence.

### Fireline target signals
- `IRWINID`
- engagement/outcome field
- line geometry

### NIROPS target signals
- `IRWINID`
- incident identifier/name
- UTC/date field
- polygon geometry

The selected source is printed for review before full loading.


In [ ]:

def normalize_colname(c):
    return re.sub(r"[^a-z0-9]+", "", str(c).lower())

def source_schema_score(row, role):
    path = row["path"]
    layer = row.get("layer")
    try:
        sample = read_source(path, layer=layer, max_features=3)
    except Exception:
        return -999, []

    cols = list(sample.columns)
    norm = {normalize_colname(c): c for c in cols}
    norm_keys = list(norm.keys())
    score = 0
    reasons = []

    if any("irwin" in x for x in norm_keys):
        score += 8
        reasons.append("IRWIN")

    geom_types = {str(x).lower() for x in sample.geom_type.dropna().unique()}

    if role == "fireline":
        if any("engag" in x for x in norm_keys):
            score += 8
            reasons.append("engagement")
        if any("featurecategory" in x or "linetype" in x or "strategicline" in x for x in norm_keys):
            score += 2
            reasons.append("line metadata")
        if any("line" in g for g in geom_types):
            score += 5
            reasons.append("line geometry")

    elif role == "nirops":
        if any("utc" in x or "dateflown" in x or "dateutc" in x for x in norm_keys):
            score += 6
            reasons.append("time")
        if any("incident" in x or "incname" in x for x in norm_keys):
            score += 3
            reasons.append("incident metadata")
        if any("polygon" in g for g in geom_types):
            score += 5
            reasons.append("polygon geometry")

    return score, reasons

def score_sources(df, role):
    rows = []
    for idx, row in df.iterrows():
        score, reasons = source_schema_score(row, role)
        r = row.to_dict()
        r["score"] = score
        r["reasons"] = ", ".join(reasons)
        rows.append(r)
    return pd.DataFrame(rows).sort_values("score", ascending=False)

fireline_scored = score_sources(fireline_sources, "fireline")
nirops_scored = score_sources(nirops_sources, "nirops")

print("FIRELINE SOURCE RANKING")
display(fireline_scored)

print("\nNIROPS SOURCE RANKING")
display(nirops_scored)

assert len(fireline_scored), "No Fireline spatial source found."
assert len(nirops_scored), "No NIROPS spatial source found."

FIRELINE_SOURCE = fireline_scored.iloc[0]
NIROPS_SOURCE = nirops_scored.iloc[0]

print("\nSELECTED FIRELINE")
print(FIRELINE_SOURCE[["path", "layer", "score", "reasons"]].to_dict())

print("\nSELECTED NIROPS")
print(NIROPS_SOURCE[["path", "layer", "score", "reasons"]].to_dict())



## Optional manual override

If the automatic ranking above chose the wrong source, set the two variables below manually and rerun from here.

Leave both as `None` to use the automatic selections.


In [ ]:

# Example:
# FIRELINE_MANUAL_PATH = Path(r"[LOCAL_PATH]")
# FIRELINE_MANUAL_LAYER = "Fireline_QAQC_Engagement"
#
# NIROPS_MANUAL_PATH = Path(r"[LOCAL_PATH]")
# NIROPS_MANUAL_LAYER = None

FIRELINE_MANUAL_PATH = None
FIRELINE_MANUAL_LAYER = None

NIROPS_MANUAL_PATH = None
NIROPS_MANUAL_LAYER = None

fireline_path = Path(FIRELINE_MANUAL_PATH) if FIRELINE_MANUAL_PATH else Path(FIRELINE_SOURCE["path"])
fireline_layer = FIRELINE_MANUAL_LAYER if FIRELINE_MANUAL_PATH else FIRELINE_SOURCE.get("layer")

nirops_path = Path(NIROPS_MANUAL_PATH) if NIROPS_MANUAL_PATH else Path(NIROPS_SOURCE["path"])
nirops_layer = NIROPS_MANUAL_LAYER if NIROPS_MANUAL_PATH else NIROPS_SOURCE.get("layer")

if pd.isna(fireline_layer):
    fireline_layer = None
if pd.isna(nirops_layer):
    nirops_layer = None

print("Using Fireline:", fireline_path, "| layer:", fireline_layer)
print("Using NIROPS :", nirops_path, "| layer:", nirops_layer)



# 8. Load the full datasets

This may take some time, especially for the Fireline dataset.


In [ ]:

print("Loading full Fireline dataset...")
fireline = read_source(fireline_path, layer=fireline_layer)
print(f"Fireline records: {len(fireline):,}")

print("\nLoading full NIROPS dataset...")
nirops = read_source(nirops_path, layer=nirops_layer)
print(f"NIROPS records: {len(nirops):,}")

print("\nMemory usage:")
print("Fireline:", human_bytes(fireline.memory_usage(deep=True).sum()))
print("NIROPS :", human_bytes(nirops.memory_usage(deep=True).sum()))



# 9. Full schema and data-quality audit


In [ ]:

def dataset_overview(gdf, name):
    print("=" * 100)
    print(name)
    print("=" * 100)
    print("Rows          :", f"{len(gdf):,}")
    print("Columns       :", len(gdf.columns))
    print("CRS           :", gdf.crs)
    print("Geometry types:", gdf.geom_type.value_counts(dropna=False).to_dict())
    print("Empty geometry:", int(gdf.geometry.is_empty.sum()))
    print("Null geometry :", int(gdf.geometry.isna().sum()))
    try:
        invalid = (~gdf.geometry.is_valid & gdf.geometry.notna()).sum()
        print("Invalid geom  :", int(invalid))
    except Exception:
        pass

    schema = pd.DataFrame({
        "column": gdf.columns,
        "dtype": [str(gdf[c].dtype) for c in gdf.columns],
        "missing_n": [int(gdf[c].isna().sum()) for c in gdf.columns],
        "missing_pct": [float(gdf[c].isna().mean() * 100) for c in gdf.columns],
        "n_unique": [
            np.nan if c == gdf.geometry.name else int(gdf[c].nunique(dropna=True))
            for c in gdf.columns
        ],
    })
    return schema

fireline_schema = dataset_overview(fireline, "FIRELINE QA/QC")
display(fireline_schema)

nirops_schema = dataset_overview(nirops, "NIROPS")
display(nirops_schema)

fireline_schema.to_csv(OUTPUT_ROOT / "fireline_schema.csv", index=False)
nirops_schema.to_csv(OUTPUT_ROOT / "nirops_schema.csv", index=False)



# 10. Detect important columns

The notebook tries to identify:

### Both
- IRWINID
- incident name
- year

### Fireline
- engagement outcome
- length
- line type

### NIROPS
- observation timestamp
- acquisition date


In [ ]:

def find_column(columns, exact_candidates=(), contains_candidates=()):
    cols = list(columns)
    norm_map = {normalize_colname(c): c for c in cols}

    for candidate in exact_candidates:
        key = normalize_colname(candidate)
        if key in norm_map:
            return norm_map[key]

    for token in contains_candidates:
        token_norm = normalize_colname(token)
        for norm, original in norm_map.items():
            if token_norm in norm:
                return original
    return None

# Fireline
FL_IRWIN = find_column(fireline.columns,
                       ["IRWINID", "IRWIN_ID", "IrwinID"],
                       ["irwin"])
FL_NAME = find_column(fireline.columns,
                      ["IncidentName", "Incident_Name", "IncName", "Inc_Name"],
                      ["incidentname", "incname"])
FL_YEAR = find_column(fireline.columns,
                      ["Year", "FireYear", "IncidentYear"],
                      ["year"])
FL_ENGAGEMENT = find_column(fireline.columns,
                            ["FirelineEngagement", "Engagement", "Fireline_Engagement"],
                            ["engagement"])
FL_LENGTH = find_column(fireline.columns,
                        ["LengthFeet", "Length_Feet", "Shape_Length", "Length"],
                        ["lengthfeet", "shapelength"])
FL_LINE_TYPE = find_column(fireline.columns,
                           ["FeatureCategory", "StrategicLineType", "LineType"],
                           ["featurecategory", "strategiclinetype", "linetype"])

# NIROPS
NR_IRWIN = find_column(nirops.columns,
                       ["IRWINID", "IRWIN_ID", "IrwinID"],
                       ["irwin"])
NR_NAME = find_column(nirops.columns,
                      ["Inc_Name", "IncidentName", "Incident_C", "IncName"],
                      ["incname", "incidentname", "incidentc"])
NR_YEAR = find_column(nirops.columns,
                      ["Year", "FireYear", "IncidentYear"],
                      ["year"])
NR_TIME = find_column(nirops.columns,
                      ["UTC", "DateUTC", "DateTimeUTC", "DateFlown"],
                      ["utc", "dateflown"])

detected = pd.DataFrame([
    ["Fireline", "IRWIN", FL_IRWIN],
    ["Fireline", "Incident name", FL_NAME],
    ["Fireline", "Year", FL_YEAR],
    ["Fireline", "Engagement", FL_ENGAGEMENT],
    ["Fireline", "Length", FL_LENGTH],
    ["Fireline", "Line type", FL_LINE_TYPE],
    ["NIROPS", "IRWIN", NR_IRWIN],
    ["NIROPS", "Incident name", NR_NAME],
    ["NIROPS", "Year", NR_YEAR],
    ["NIROPS", "Timestamp", NR_TIME],
], columns=["dataset", "role", "column"])

display(detected)

assert FL_IRWIN is not None, "Could not detect Fireline IRWINID column."
assert NR_IRWIN is not None, "Could not detect NIROPS IRWINID column."
assert FL_ENGAGEMENT is not None, "Could not detect Fireline engagement column."



# 11. Normalize IRWIN identifiers

IRWIN identifiers may differ only by braces, hyphens, case, or whitespace.

Normalization:

```text
{ABC-123} -> ABC123
abc-123   -> ABC123
```

No semantic alteration is performed.


In [ ]:

def normalize_irwin(value):
    if pd.isna(value):
        return None
    s = str(value).strip().upper()
    if not s or s in {"NONE", "NAN", "NULL", "<NA>"}:
        return None
    s = re.sub(r"[^A-Z0-9]", "", s)
    return s or None

fireline["_irwin_norm"] = fireline[FL_IRWIN].map(normalize_irwin)
nirops["_irwin_norm"] = nirops[NR_IRWIN].map(normalize_irwin)

def id_quality(df, raw_col, norm_col, name):
    print(name)
    print("Raw non-null        :", f"{df[raw_col].notna().sum():,}")
    print("Normalized non-null :", f"{df[norm_col].notna().sum():,}")
    print("Unique normalized   :", f"{df[norm_col].nunique(dropna=True):,}")
    print("Missing normalized  :", f"{df[norm_col].isna().sum():,}")
    print()

id_quality(fireline, FL_IRWIN, "_irwin_norm", "FIRELINE IRWIN")
id_quality(nirops, NR_IRWIN, "_irwin_norm", "NIROPS IRWIN")



# 12. Determine incident year

If a dedicated year field does not exist, the notebook attempts to derive year from the available observation date/timestamp.

For Fireline, a year field is strongly preferred.  
For NIROPS, year can safely be derived from the acquisition timestamp for inventory purposes.


In [ ]:

def coerce_year(series):
    numeric = pd.to_numeric(series, errors="coerce")
    # Handle actual year values.
    out = numeric.where((numeric >= 1900) & (numeric <= 2100))
    return out.astype("Int64")

if FL_YEAR is not None:
    fireline["_year"] = coerce_year(fireline[FL_YEAR])
else:
    fireline["_year"] = pd.Series(pd.NA, index=fireline.index, dtype="Int64")

if NR_YEAR is not None:
    nirops["_year"] = coerce_year(nirops[NR_YEAR])
else:
    nirops["_year"] = pd.Series(pd.NA, index=nirops.index, dtype="Int64")

if NR_TIME is not None:
    nirops["_time"] = pd.to_datetime(nirops[NR_TIME], errors="coerce", utc=True)
    missing_year = nirops["_year"].isna()
    nirops.loc[missing_year, "_year"] = nirops.loc[missing_year, "_time"].dt.year.astype("Int64")
else:
    nirops["_time"] = pd.NaT

print("Fireline year distribution:")
display(fireline["_year"].value_counts(dropna=False).sort_index())

print("NIROPS year distribution:")
display(nirops["_year"].value_counts(dropna=False).sort_index())



# 13. Incident-level inventories

This is the first key joinability output.

The notebook constructs one row per normalized IRWIN ID for each dataset.


In [ ]:

def first_nonempty(series):
    for v in series:
        if pd.notna(v) and str(v).strip():
            return v
    return None

def incident_inventory(df, irwin_col, name_col=None):
    agg = {
        "n_records": (irwin_col, "size"),
        "year_min": ("_year", "min"),
        "year_max": ("_year", "max"),
    }
    if name_col is not None:
        agg["incident_name"] = (name_col, first_nonempty)

    inv = (
        df[df[irwin_col].notna()]
        .groupby(irwin_col, dropna=False)
        .agg(**agg)
        .reset_index()
        .rename(columns={irwin_col: "_irwin_norm"})
    )
    return inv

fireline_incidents = incident_inventory(fireline, "_irwin_norm", FL_NAME)
nirops_incidents = incident_inventory(nirops, "_irwin_norm", NR_NAME)

print("Unique Fireline incidents:", f"{len(fireline_incidents):,}")
print("Unique NIROPS incidents  :", f"{len(nirops_incidents):,}")

display(fireline_incidents.head())
display(nirops_incidents.head())



# 14. Exact IRWINID join

This is the primary, trusted join.

No fuzzy logic is used here.


In [ ]:

fire_ids = set(fireline_incidents["_irwin_norm"])
nirops_ids = set(nirops_incidents["_irwin_norm"])

matched_ids = fire_ids & nirops_ids
fire_only_ids = fire_ids - nirops_ids
nirops_only_ids = nirops_ids - fire_ids

print("=" * 80)
print("EXACT IRWIN JOIN")
print("=" * 80)
print(f"Fireline unique IRWIN incidents : {len(fire_ids):,}")
print(f"NIROPS unique IRWIN incidents   : {len(nirops_ids):,}")
print(f"Exact matched incidents         : {len(matched_ids):,}")
print(f"Fireline unmatched IRWIN        : {len(fire_only_ids):,}")
print(f"NIROPS unmatched IRWIN          : {len(nirops_only_ids):,}")
print()
print(f"Fireline incident coverage      : {len(matched_ids)/len(fire_ids)*100:.2f}%")
print(f"NIROPS incident coverage        : {len(matched_ids)/len(nirops_ids)*100:.2f}%")

incident_crosswalk = (
    fireline_incidents
    .merge(
        nirops_incidents,
        on="_irwin_norm",
        how="inner",
        suffixes=("_fireline", "_nirops")
    )
)

incident_crosswalk["match_method"] = "exact_irwin"

display(incident_crosswalk.head(20))



# 15. Restrict Fireline records to matched incidents

This determines how much labeled suppression-line data survives the incident join.


In [ ]:

matched_fireline = fireline[fireline["_irwin_norm"].isin(matched_ids)].copy()
matched_nirops = nirops[nirops["_irwin_norm"].isin(matched_ids)].copy()

print("Matched Fireline records:", f"{len(matched_fireline):,}")
print("Matched NIROPS records  :", f"{len(matched_nirops):,}")

print("\nFireline engagement values:")
display(matched_fireline[FL_ENGAGEMENT].value_counts(dropna=False))



# 16. Normalize engagement labels

The code maps common spelling/format variants into:

- `Held`
- `Burned Over`
- `Not Engaged`
- `Other/Unknown`

Original labels are preserved.


In [ ]:

def normalize_engagement(v):
    if pd.isna(v):
        return "Other/Unknown"
    s = re.sub(r"\s+", " ", str(v).strip().lower())
    s2 = re.sub(r"[^a-z]+", "", s)

    if "notengag" in s2 or "unengag" in s2:
        return "Not Engaged"
    if "burn" in s2 and "over" in s2:
        return "Burned Over"
    if s2 == "held" or "hold" in s2 or "held" in s2:
        return "Held"
    return "Other/Unknown"

matched_fireline["_engagement_norm"] = matched_fireline[FL_ENGAGEMENT].map(normalize_engagement)

engagement_counts = (
    matched_fireline["_engagement_norm"]
    .value_counts(dropna=False)
    .rename_axis("engagement")
    .reset_index(name="n_records")
)

engagement_counts["pct_records"] = (
    engagement_counts["n_records"] / engagement_counts["n_records"].sum() * 100
)

display(engagement_counts)



# 17. Compute fireline lengths robustly

If a trustworthy length field exists, it is inspected.

For consistency across datasets, this notebook also computes geometric line lengths using an appropriate projected CRS.

### Strategy
- If the source CRS is projected in meters, use geometry length directly.
- Otherwise project each feature to a CONUS equal-distance-ish metric CRS (`EPSG:5070`) and compute meters.

The audit reports:
- total length in km
- length by engagement class
- length by year


In [ ]:

def compute_length_m(gdf):
    if gdf.empty:
        return pd.Series(dtype=float, index=gdf.index)

    # EPSG:5070 is appropriate for CONUS-scale area/distance calculations.
    try:
        projected = gdf.to_crs(5070)
        return projected.geometry.length.astype(float)
    except Exception as exc:
        raise RuntimeError(f"Could not compute metric line length: {exc}")

matched_fireline["_length_m_geom"] = compute_length_m(matched_fireline)
matched_fireline["_length_km_geom"] = matched_fireline["_length_m_geom"] / 1000.0

if FL_LENGTH:
    matched_fireline["_source_length_numeric"] = pd.to_numeric(
        matched_fireline[FL_LENGTH], errors="coerce"
    )
    print("Detected source length field:", FL_LENGTH)
    print(matched_fireline["_source_length_numeric"].describe())
else:
    print("No source length field detected; using geometry-based metric length only.")

length_summary = (
    matched_fireline
    .groupby("_engagement_norm", dropna=False)
    .agg(
        n_records=("_engagement_norm", "size"),
        length_km=("_length_km_geom", "sum"),
        mean_segment_km=("_length_km_geom", "mean"),
        median_segment_km=("_length_km_geom", "median"),
    )
    .reset_index()
    .rename(columns={"_engagement_norm": "engagement"})
)

length_summary["pct_total_length"] = (
    length_summary["length_km"] / length_summary["length_km"].sum() * 100
)

print("TOTAL MATCHED FIRELINE LENGTH:", f"{matched_fireline['_length_km_geom'].sum():,.2f} km")
display(length_summary)



# 18. Matched incident outcome summary

Count outcomes at the incident level as well as at the record level.


In [ ]:

incident_outcomes = (
    matched_fireline
    .groupby(["_irwin_norm", "_engagement_norm"], dropna=False)
    .agg(
        n_line_records=("_engagement_norm", "size"),
        length_km=("_length_km_geom", "sum"),
    )
    .reset_index()
)

incident_outcome_pivot = (
    incident_outcomes
    .pivot_table(
        index="_irwin_norm",
        columns="_engagement_norm",
        values="length_km",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

display(incident_outcome_pivot.head(20))



# 19. NIROPS observation counts per matched fire


In [ ]:

if NR_TIME is not None:
    # Treat exact duplicate timestamps within an incident as one observation time.
    nirops_obs = (
        matched_nirops
        .groupby("_irwin_norm")
        .agg(
            n_records=("_irwin_norm", "size"),
            n_observation_times=("_time", lambda s: s.dropna().nunique()),
            first_observation=("_time", "min"),
            last_observation=("_time", "max"),
            year_min=("_year", "min"),
            year_max=("_year", "max"),
        )
        .reset_index()
    )
    nirops_obs["observation_span_days"] = (
        (nirops_obs["last_observation"] - nirops_obs["first_observation"])
        .dt.total_seconds() / 86400
    )
else:
    nirops_obs = (
        matched_nirops.groupby("_irwin_norm")
        .size().rename("n_records").reset_index()
    )
    nirops_obs["n_observation_times"] = np.nan

display(nirops_obs.describe(include="all").T)

thresholds = [3, 5, 10, 20]
threshold_rows = []
for t in thresholds:
    if "n_observation_times" in nirops_obs:
        n = int((nirops_obs["n_observation_times"] >= t).sum())
        threshold_rows.append({
            "minimum_observations": t,
            "n_matched_incidents": n,
            "pct_matched_incidents": n / len(nirops_obs) * 100 if len(nirops_obs) else np.nan,
        })

observation_thresholds = pd.DataFrame(threshold_rows)
display(observation_thresholds)



# 20. NIROPS temporal-gap audit

For each matched incident, sort unique observation timestamps and calculate consecutive time gaps.

This is critical for deciding whether dynamic fire-approach intervals are usable.


In [ ]:

gap_rows = []

if NR_TIME is not None:
    for irwin, grp in matched_nirops.groupby("_irwin_norm"):
        times = pd.Series(grp["_time"].dropna().unique()).sort_values()
        if len(times) < 2:
            continue

        diffs_hours = times.diff().dt.total_seconds().div(3600.0).dropna()

        for gap in diffs_hours:
            gap_rows.append({
                "_irwin_norm": irwin,
                "gap_hours": float(gap),
            })

nirops_gaps = pd.DataFrame(gap_rows)

if len(nirops_gaps):
    print(nirops_gaps["gap_hours"].describe(percentiles=[.1,.25,.5,.75,.9,.95,.99]))

    gap_bins = pd.cut(
        nirops_gaps["gap_hours"],
        bins=[0, 12, 24, 36, 48, 72, 96, np.inf],
        right=True,
        include_lowest=True,
    )
    gap_summary = (
        gap_bins.value_counts(sort=False)
        .rename_axis("gap_bin")
        .reset_index(name="n_gaps")
    )
    gap_summary["pct_gaps"] = gap_summary["n_gaps"] / gap_summary["n_gaps"].sum() * 100
    display(gap_summary)
else:
    gap_summary = pd.DataFrame()
    print("No temporal gaps could be calculated.")



# 21. Year-by-year exact overlap

This is useful for potential temporal train/validation/test splits.


In [ ]:

# Incident-year association based on each dataset's inventory.
fl_year_inc = (
    fireline[fireline["_irwin_norm"].notna() & fireline["_year"].notna()]
    [["_irwin_norm", "_year"]]
    .drop_duplicates()
)

nr_year_inc = (
    nirops[nirops["_irwin_norm"].notna() & nirops["_year"].notna()]
    [["_irwin_norm", "_year"]]
    .drop_duplicates()
)

year_rows = []
years = sorted(set(fl_year_inc["_year"].dropna().astype(int)) | set(nr_year_inc["_year"].dropna().astype(int)))

for year in years:
    fl_ids_y = set(fl_year_inc.loc[fl_year_inc["_year"] == year, "_irwin_norm"])
    nr_ids_y = set(nr_year_inc.loc[nr_year_inc["_year"] == year, "_irwin_norm"])
    m = fl_ids_y & nr_ids_y
    year_rows.append({
        "year": int(year),
        "fireline_incidents": len(fl_ids_y),
        "nirops_incidents": len(nr_ids_y),
        "exact_matched_incidents": len(m),
        "fireline_match_pct": len(m) / len(fl_ids_y) * 100 if fl_ids_y else np.nan,
        "nirops_match_pct": len(m) / len(nr_ids_y) * 100 if nr_ids_y else np.nan,
    })

year_overlap = pd.DataFrame(year_rows)
display(year_overlap)



# 22. Conservative fallback-match candidates

Exact IRWIN matching is authoritative.

For incidents missing an exact match, this section creates **review candidates only** using normalized incident name and year.

Nothing here is added to the trusted crosswalk automatically.

A high similarity score means:
> "worth manually checking"

not:
> "confirmed match."


In [ ]:

def normalize_incident_name(v):
    if pd.isna(v):
        return None
    s = str(v).upper().strip()
    # Remove common generic terms that do not help distinguish incidents.
    s = re.sub(r"\b(FIRE|INCIDENT|COMPLEX)\b", " ", s)
    s = re.sub(r"[^A-Z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s or None

fallback_candidates = []

if FL_NAME is not None and NR_NAME is not None:
    fl_unmatched = (
        fireline[fireline["_irwin_norm"].isin(fire_only_ids)]
        [["_irwin_norm", FL_NAME, "_year"]]
        .drop_duplicates()
        .copy()
    )
    nr_unmatched = (
        nirops[nirops["_irwin_norm"].isin(nirops_only_ids)]
        [["_irwin_norm", NR_NAME, "_year"]]
        .drop_duplicates()
        .copy()
    )

    fl_unmatched["_name_norm"] = fl_unmatched[FL_NAME].map(normalize_incident_name)
    nr_unmatched["_name_norm"] = nr_unmatched[NR_NAME].map(normalize_incident_name)

    # Group NIROPS candidates by year to reduce false matches.
    nr_by_year = {
        year: grp.dropna(subset=["_name_norm"]).copy()
        for year, grp in nr_unmatched.groupby("_year", dropna=False)
    }

    for _, row in tqdm(fl_unmatched.dropna(subset=["_name_norm"]).iterrows(),
                       total=fl_unmatched["_name_norm"].notna().sum(),
                       desc="Fallback name matching"):
        year = row["_year"]
        pool = nr_by_year.get(year)
        if pool is None or pool.empty:
            continue

        query = row["_name_norm"]
        choices = pool["_name_norm"].tolist()

        match = process.extractOne(query, choices, scorer=fuzz.token_sort_ratio)
        if match is None:
            continue

        matched_name, score, idx_in_choices = match
        candidate_row = pool.iloc[idx_in_choices]

        if score >= 70:
            fallback_candidates.append({
                "fireline_irwin": row["_irwin_norm"],
                "fireline_name": row[FL_NAME],
                "nirops_irwin": candidate_row["_irwin_norm"],
                "nirops_name": candidate_row[NR_NAME],
                "year": year,
                "name_similarity": score,
                "status": "REVIEW_ONLY_NOT_ACCEPTED",
            })

fallback_candidates = pd.DataFrame(fallback_candidates)

if len(fallback_candidates):
    fallback_candidates = fallback_candidates.sort_values(
        ["name_similarity"], ascending=False
    )
    display(fallback_candidates.head(100))
else:
    print("No fallback name/year candidates >= 70 similarity.")



# 23. Save audit outputs


In [ ]:

# Core crosswalks.
incident_crosswalk.to_csv(OUTPUT_ROOT / "incident_crosswalk_exact_irwin.csv", index=False)
fireline_incidents[fireline_incidents["_irwin_norm"].isin(fire_only_ids)].to_csv(
    OUTPUT_ROOT / "unmatched_fireline_incidents.csv", index=False
)
nirops_incidents[nirops_incidents["_irwin_norm"].isin(nirops_only_ids)].to_csv(
    OUTPUT_ROOT / "unmatched_nirops_incidents.csv", index=False
)

if len(fallback_candidates):
    fallback_candidates.to_csv(
        OUTPUT_ROOT / "fallback_match_candidates_REVIEW_ONLY.csv", index=False
    )

# Outcome / temporal summaries.
engagement_counts.to_csv(OUTPUT_ROOT / "outcome_record_summary.csv", index=False)
length_summary.to_csv(OUTPUT_ROOT / "outcome_length_summary.csv", index=False)
incident_outcomes.to_csv(OUTPUT_ROOT / "incident_outcome_summary.csv", index=False)
nirops_obs.to_csv(OUTPUT_ROOT / "nirops_observation_summary.csv", index=False)
observation_thresholds.to_csv(OUTPUT_ROOT / "nirops_observation_thresholds.csv", index=False)
year_overlap.to_csv(OUTPUT_ROOT / "year_overlap_summary.csv", index=False)

if len(nirops_gaps):
    nirops_gaps.to_csv(OUTPUT_ROOT / "nirops_temporal_gaps.csv", index=False)
if len(gap_summary):
    gap_summary.to_csv(OUTPUT_ROOT / "nirops_temporal_gap_summary.csv", index=False)

print("Saved outputs to:", OUTPUT_ROOT)
for p in sorted(OUTPUT_ROOT.iterdir()):
    if p.is_file():
        print(" -", p.name, human_bytes(p.stat().st_size))



# 24. Go / no-go assessment

This is deliberately heuristic, not a statistical theorem.

Suggested interpretation:

- **>250 exact matched fires** → very strong data basis
- **100–250** → good multi-incident study
- **50–100** → potentially viable, but narrower claims
- **<50** → reconsider the dynamic multi-fire framing

The final decision also depends on:
- Held/Burned Over balance
- amount of matched line length
- number of NIROPS observations per fire
- temporal-gap distribution
- geographic diversity


In [ ]:

n_match = len(matched_ids)

if n_match > 250:
    go_rating = "VERY STRONG GO"
    go_reason = "More than 250 exact IRWIN-matched incidents."
elif n_match >= 100:
    go_rating = "GO"
    go_reason = "At least 100 exact IRWIN-matched incidents."
elif n_match >= 50:
    go_rating = "CONDITIONAL GO"
    go_reason = "50–99 exact matches; likely requires narrower claims and careful validation."
else:
    go_rating = "RECONSIDER"
    go_reason = "Fewer than 50 exact matched incidents."

print("=" * 80)
print("PRELIMINARY JOINABILITY ASSESSMENT")
print("=" * 80)
print("Assessment:", go_rating)
print("Reason    :", go_reason)
print()
print("Exact matched incidents:", f"{n_match:,}")
print("Matched Fireline records:", f"{len(matched_fireline):,}")
print("Matched NIROPS records  :", f"{len(matched_nirops):,}")
print("Matched line length     :", f"{matched_fireline['_length_km_geom'].sum():,.2f} km")
print()
print("Engagement record counts:")
for _, r in engagement_counts.iterrows():
    print(f"  {r['engagement']:<18} {int(r['n_records']):>12,} ({r['pct_records']:.2f}%)")

if len(observation_thresholds):
    print("\nNIROPS matched-fire observation thresholds:")
    for _, r in observation_thresholds.iterrows():
        print(
            f"  >= {int(r['minimum_observations']):>2} observations: "
            f"{int(r['n_matched_incidents']):>5,} fires "
            f"({r['pct_matched_incidents']:.1f}%)"
        )



# 25. Generate `JOINABILITY_REPORT.md`

This report is intended to be readable outside the notebook and can later be committed with the project.


In [ ]:
def df_to_md(df, max_rows=100):
    if df is None or len(df) == 0:
        return "_No rows._"
    try:
        return df.head(max_rows).to_markdown(index=False)
    except Exception:
        return df.head(max_rows).to_string(index=False)


# Prepare NIROPS observation summary separately
nirops_obs_summary = (
    nirops_obs
    .describe(include="all")
    .T
    .reset_index()
    .rename(columns={"index": "metric"})
)


report = f"""# Fireline QA/QC × NIROPS Joinability Report

Generated: {datetime.now(timezone.utc).isoformat()}

## Sources

### Fireline
- Source archive: `{FIRELINE_CORE_ZIP}`
- Loaded source: `{fireline_path}`
- Layer: `{fireline_layer}`
- Records: {len(fireline):,}
- CRS: `{fireline.crs}`

### NIROPS
- Source archive: `{NIROPS_CORE_ZIP}`
- Loaded source: `{nirops_path}`
- Layer: `{nirops_layer}`
- Records: {len(nirops):,}
- CRS: `{nirops.crs}`

## Exact IRWIN Join

- Fireline unique normalized IRWIN incidents: **{len(fire_ids):,}**
- NIROPS unique normalized IRWIN incidents: **{len(nirops_ids):,}**
- Exact matched incidents: **{len(matched_ids):,}**
- Fireline unmatched incidents: **{len(fire_only_ids):,}**
- NIROPS unmatched incidents: **{len(nirops_only_ids):,}**
- Fireline incident coverage: **{len(matched_ids)/len(fire_ids)*100:.2f}%**
- NIROPS incident coverage: **{len(matched_ids)/len(nirops_ids)*100:.2f}%**

## Matched Fireline Data

- Matched Fireline records: **{len(matched_fireline):,}**
- Matched total geometry-based line length: **{matched_fireline['_length_km_geom'].sum():,.2f} km**

### Engagement outcomes

{df_to_md(length_summary)}

## NIROPS observations for matched incidents

{df_to_md(observation_thresholds)}

### Observation summary

{df_to_md(nirops_obs_summary, max_rows=100)}

## NIROPS temporal gaps

{df_to_md(gap_summary)}

## Year overlap

{df_to_md(year_overlap)}

## Fallback matching

Fallback name/year matching is **review-only** and is not included in the trusted exact crosswalk.

Fallback candidates produced: **{len(fallback_candidates):,}**

## Preliminary assessment

**{go_rating}**

{go_reason}

## Scientific interpretation

The exact IRWIN match is the trusted basis for the initial dynamic study.

The next stage should NOT immediately train ML. The next technical stage should:

1. inspect the spatial relationship between NIROPS progression and matched Fireline features on a sample of incidents;
2. determine whether Fireline features should be segmented and at what spatial resolution;
3. define an interval-censored fire-approach/challenge window;
4. quantify how many Held and Burned Over line segments can be temporally corroborated by NIROPS;
5. only then begin extracting weather, terrain, and pre-fire fuel variables.

## Important limitations

- Fireline engagement labels are spatial QA/QC outcomes, not necessarily direct field-observed hold/failure events.
- NIROPS acquisition is intermittent, so exact line-contact time is not observed.
- Raw suppression effort variables are incomplete.
- Nearby line segments within the same incident are not independent.
- Future modeling must split by incident, never randomly across line segments.
"""

report_path = OUTPUT_ROOT / "JOINABILITY_REPORT.md"
report_path.write_text(report, encoding="utf-8")

print(report)
print("\nSaved report:", report_path)


# 26. Final file inventory

This last cell shows all generated audit artifacts.


In [ ]:

print("AUDIT OUTPUTS")
print("=" * 80)

for p in sorted(OUTPUT_ROOT.rglob("*")):
    if p.is_file():
        print(f"{p.name:<50} {human_bytes(p.stat().st_size):>12}")

print("\nDone.")
print("Review JOINABILITY_REPORT.md before proceeding to dynamic feature engineering.")
